# A-ADP — Preliminary Run on CT-RATE (Colab)

Runs the **whole pipeline** end-to-end on a ~45 GB slice of CT-RATE v2:
download → train (small config, LoRA) → VTCB eval → figures, all persisted to a
**second account's Google Drive** via rclone, while the runtime uses **this
account's Colab Pro** GPU.

### Accounts
- **This account (A):** has Colab Pro → runs the GPU runtime.
- **Second account (B):** has the 70 GB Drive → stores the data + outputs.

### One-time prerequisites (add these as Colab **Secrets** — the 🔑 icon)
1. **`HF_TOKEN`** for the gated CT-RATE dataset (accept the license on the HF page first).
2. **`GH_TOKEN`** — a GitHub Personal Access Token with read access to the (private)
   repo. GitHub → Settings → Developer settings → Fine-grained tokens → select the
   repo → Contents: Read-only.
3. **rclone token for Account B.** On your laptop (with rclone installed) run
   `rclone authorize "drive"`, sign in as **Account B**, and copy the whole JSON
   blob it prints (`{"access_token":...}`). You'll paste it in Step 2.

> Set the runtime to GPU: **Runtime → Change runtime type → GPU** (T4/L4/A100).
> Enable **Notebook access** on each secret so the cells can read them.


## Step 0 — Clone the (private) repo and install

In [ ]:
!nvidia-smi -L

from google.colab import userdata
GH_TOKEN = userdata.get("GH_TOKEN")   # private-repo PAT from the Secrets panel

!git clone https://{GH_TOKEN}@github.com/Tech-sam-90/3d_compression.git
%cd 3d_compression
!pip -q install -e .    # setup.py now pulls SimpleITK, nibabel, timm, datasets, etc.
!pip -q install tabulate


## Step 1 — HuggingFace token (gated CT-RATE)

In [ ]:
import os
# Option A: Colab secret named HF_TOKEN (recommended) — key icon in the sidebar.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXX"  # or paste it here
assert os.environ.get("HF_TOKEN", "").startswith("hf_"), "Set a valid HF token"
print("HF token set.")


## Step 2 — Connect Account B's Drive via rclone (no FUSE mount)

Colab runtimes generally can't do a FUSE `rclone mount`, so we use `rclone copy`
instead: data and outputs live on the fast **local** disk during the session and
are pushed to **Account B** with `rclone copy` (files land under B's quota, not
this Pro account). Paste the token JSON from `rclone authorize "drive"` (run on
your laptop as **Account B**) below.


In [ ]:
import pathlib

RCLONE_TOKEN = '{"access_token":"PASTE_FROM_rclone_authorize","token_type":"Bearer","refresh_token":"...","expiry":"..."}'

# Install rclone and write a config for Account B's Drive (remote name: driveB).
!curl -s https://rclone.org/install.sh | sudo bash

conf = f'''[driveB]
type = drive
scope = drive
token = {RCLONE_TOKEN}
'''
cfg = pathlib.Path.home() / ".config/rclone/rclone.conf"
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text(conf)

# Verify the remote works — lists Account B's top-level folders (no mount needed).
!rclone lsd driveB: | head


Working folders: **local** during the session, synced to Account B via rclone.

In [ ]:
import os
DRIVE_REMOTE = "driveB:aadp_prelim"    # persisted on Account B (via rclone copy)
DATA_DIR = "/content/ctrate"           # fast local working copy of the subset
OUT      = "/content/outputs"
CKPT_DIR = f"{OUT}/checkpoints"
RES_DIR  = f"{OUT}/results"
FIG_DIR  = f"{OUT}/figures"
for d in (DATA_DIR, CKPT_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
print("Local working dir:", OUT, "| will sync to", DRIVE_REMOTE)


## Run configuration — **DRY_RUN toggle**

Leave `DRY_RUN = True` for a fast (~2 GB / 20 volumes, ~15 min) end-to-end check
that the mount + auth + pipeline all work. Once it's green, set `DRY_RUN = False`
and re-run from Step 3 for the full ~45 GB run.


In [ ]:
DRY_RUN = True   # True: ~2 GB smoke test | False: full ~45 GB run

if DRY_RUN:
    MAX_GB, N_TRAIN, N_VAL = 2.0, 20, 6
    VTCB_BUDGETS, VTCB_VAL, PROBE_TRAIN, PROBE_STEPS = [16, 64], 6, 16, 40
else:
    MAX_GB, N_TRAIN, N_VAL = 45.0, 300, 20
    VTCB_BUDGETS, VTCB_VAL, PROBE_TRAIN, PROBE_STEPS = [16, 64], 16, 64, 150

print(f"DRY_RUN={DRY_RUN} | data≈{MAX_GB} GB | train/val={N_TRAIN}/{N_VAL} "
      f"| VTCB budgets={VTCB_BUDGETS}")


## Step 3 — Download the CT-RATE subset (local) and sync to Account B

Size-budgeted download (v2 fixed volumes, non-chest scans excluded) to the fast
local disk, then `rclone copy` pushes it to B for persistence. `max_gb` is what
stops it (10% reserved for validation); `n_train`/`n_valid` are safety caps.
To resume next session, run the commented restore line first.


In [ ]:
from aadp.data.ctrate_dataset import download_subset_to_disk

# Resume helper (optional): pull a previously-saved subset back from B first,
# so the download only tops up what's missing.
# !rclone copy {DRIVE_REMOTE}/ctrate {DATA_DIR} -P --transfers=8

download_subset_to_disk(
    local_data_dir=DATA_DIR,
    max_gb=MAX_GB,        # from the DRY_RUN cell (~2 GB smoke, or 45 GB full)
    valid_ratio=0.1,
    n_train=8000,         # safety upper bounds (size budget binds first)
    n_valid=800,
    max_workers=8,
)

# Persist the subset to Account B (skip this line if you'd rather re-download
# next session than wait for the upload).
!rclone copy {DATA_DIR} {DRIVE_REMOTE}/ctrate -P --transfers=8


## Step 4 — Build the model (small config + LoRA) and train

In [ ]:
import torch
from aadp.models.vlm import MedVLM
from aadp.data.ctrate_dataset import CTRATEDataset
from aadp.data.multitask_sampler import MultiTaskCTRATEDataset
from aadp.training.trainer import Trainer
from aadp.training.losses import NextTokenLoss
from aadp.training.scheduler import get_cosine_schedule_with_warmup

DEVICE = "cuda"
LORA = {"enabled": True, "r": 16, "alpha": 32,
        "target_modules": ["q_proj", "v_proj"], "dropout": 0.05}

# Small, fast, memory-safe config for a preliminary run. Scale up later.
model = MedVLM(
    vit_model_name="vit_tiny_patch16_224",
    llm_model_name="facebook/opt-125m",
    instruction_encoder_model="facebook/opt-125m",
    vit_resize_to=224,          # keep slice memory in check
    llm_frozen=False, llm_lora=LORA,
    num_latents=32, num_tokens=64,
    device=DEVICE,
)

# shuffle=False so the on-disk (first-N) volumes line up with iteration order.
# N_TRAIN / N_VAL come from the DRY_RUN cell above.
train_base = CTRATEDataset("train", local_data_dir=DATA_DIR, shuffle=False, max_samples=N_TRAIN)
val_base   = CTRATEDataset("valid", local_data_dir=DATA_DIR, shuffle=False, max_samples=N_VAL)
train_ds = MultiTaskCTRATEDataset(train_base)   # T1-T4 instruction mix per sample
val_ds   = MultiTaskCTRATEDataset(val_base)

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
GRAD_ACCUM = 8
num_steps = max(1, N_TRAIN // GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=5, num_training_steps=num_steps)

config = dict(
    batch_size=1, gradient_accumulation_steps=GRAD_ACCUM, max_grad_norm=1.0,
    num_epochs=1, val_every_n_steps=10**9, save_every_n_steps=10**9,
    checkpoint_dir=CKPT_DIR, use_attn_loss=False, mixed_precision=True,
    patience=5, max_report_length=256,
)
trainer = Trainer(model, optimizer, scheduler, NextTokenLoss(),
                  train_ds, val_ds, config=config, device=torch.device(DEVICE), use_wandb=False)

history = trainer.train()
trainer.save_checkpoint("checkpoint_prelim.pt")
val_metrics = trainer.validate()
print("final train loss:", history["train_loss"][-1] if history["train_loss"] else None,
      "| val loss:", val_metrics["val_loss"])


Training-loss curve → save to Account B.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

losses = history["train_loss"]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(losses) + 1), losses, marker="o", ms=3)
plt.xlabel("optimizer step"); plt.ylabel("train loss (next-token CE)")
plt.title("A-ADP preliminary training loss"); plt.grid(True, ls="--", alpha=0.5)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/train_loss.png", dpi=120)
print("saved", f"{FIG_DIR}/train_loss.png")


## Step 5 — VTCB evaluation (compression-vs-quality)

Sweeps two token budgets and runs the report-generation (T1) and abnormality
classification (T2, trained linear probe) tasks. T3/T4 need RadGenome / Total-
Segmentator and are skipped here. Metric-model downloads (RadGraph/RaTEScore)
happen on first use, so this cell can take a few minutes.


In [ ]:
from aadp.evaluation.benchmarks.vtcb import VTCBRunner

vtcb_val   = CTRATEDataset("valid", local_data_dir=DATA_DIR, shuffle=False, max_samples=VTCB_VAL)
probe_train = CTRATEDataset("train", local_data_dir=DATA_DIR, shuffle=False, max_samples=PROBE_TRAIN)

runner = VTCBRunner(
    model=model, val_dataset=vtcb_val,
    token_budgets=VTCB_BUDGETS,      # small sweep for the prelim
    primary_budget=64, batch_size=1, max_samples=VTCB_VAL,
    max_new_tokens=64, device=DEVICE, results_dir=RES_DIR,
    probe_train_dataset=probe_train, probe_max_steps=PROBE_STEPS,
)
results = runner.run(model_name="aadp_prelim")

# Compression-quality curves (metric vs M) → save to B.
VTCBRunner.plot_compression_curves(
    {"aadp_prelim": f"{RES_DIR}/aadp_prelim_vtcb.json"},
    metric_names=["auroc_macro", "f1_macro", "radgraph_f1"],
    save_dir=FIG_DIR,
)
print("VTCB results + curves saved under:", OUT)


## Step 6 — Instruction-conditioned attention map (one sample)

In [ ]:
from aadp.visualization.attention_maps import visualize_attention

vol, report, labels, pid = next(iter(
    CTRATEDataset("valid", local_data_dir=DATA_DIR, shuffle=False, max_samples=1)))
instr = "Describe the location of the finding in this CT scan."

model.eval()
with torch.no_grad():
    _, attn = runner._forward_features(vol.unsqueeze(0), [instr])  # (1, D) slice attention

visualize_attention(vol, attn[0].cpu(), instr, save_path=f"{FIG_DIR}/attention_{pid}.png")
print("saved attention map for", pid)


## Step 7 — One-page summary for the update

In [ ]:
import json, pandas as pd

with open(f"{RES_DIR}/aadp_prelim_vtcb.json") as f:
    vt = json.load(f)
at64 = vt["results"].get("64", {})

rows = [
    ("volumes used (train / val)", f"{N_TRAIN} / {N_VAL}"),
    ("final train loss", round(history["train_loss"][-1], 3) if history["train_loss"] else None),
    ("val loss", round(val_metrics["val_loss"], 3)),
    ("trainable params", sum(p.numel() for p in model.parameters() if p.requires_grad)),
    ("VTCB M=64 AUROC (macro)", at64.get("abnormality_classification", {}).get("auroc_macro")),
    ("VTCB M=64 RadGraph-F1", at64.get("report_generation", {}).get("radgraph_f1")),
]
df = pd.DataFrame(rows, columns=["metric", "value"])
df.to_csv(f"{RES_DIR}/summary.csv", index=False)
try:
    with open(f"{RES_DIR}/summary.md", "w") as f:
        f.write(df.to_markdown(index=False))
except Exception as e:
    print("(summary.md skipped:", e, ")")
print(df.to_string(index=False))
print("\nLocal outputs in:", OUT, "— run Step 8 to sync them to Account B.")


## Step 8 — Sync outputs (and confirm data) on Account B

In [ ]:
# Push checkpoints / results / figures to Account B for persistence.
!rclone copy {OUT} {DRIVE_REMOTE}/outputs -P
print("\nOn Account B under", DRIVE_REMOTE + ":")
!rclone tree {DRIVE_REMOTE} --level 2 2>/dev/null || rclone lsd {DRIVE_REMOTE}


**What's on Account B when this finishes** (`driveB:aadp_prelim/`)
- `ctrate/` — the CT-RATE v2 subset (reusable next session).
- `outputs/checkpoints/checkpoint_prelim.pt` — projector + LoRA weights.
- `outputs/results/` — `aadp_prelim_vtcb.json`, `summary.csv`, `summary.md`.
- `outputs/figures/` — `train_loss.png`, VTCB curves, one attention map.

Talking points: the full instruction-conditioned pipeline (frozen ViT → two-stage
FiLM projector → LoRA LLM) trains and evaluates end-to-end on real CT-RATE, with
the VTCB compression-budget sweep and instruction-steered slice attention working.
Scale up (vit_base / opt-1.3b, more volumes, more budgets, add RadGenome/TotalSeg
for T3/T4) for the full study.
